# RAG System - Interactive Query Interface

This notebook allows you to interactively query your PDF documents using the RAG pipeline.

## 1. Setup and Imports

In [1]:
import os
import sys
import glob
from src.pdf_processor import process_pdf
from src.rag_pipeline import RAGPipeline
from config import CHUNK_SIZE, OVERLAP, PDF_DIR, OUTPUT_DIR, EMBEDDING_MODEL

print(f"Configuration:")
print(f"- PDF Directory: {PDF_DIR}")
print(f"- Embedding Model: {EMBEDDING_MODEL}")
print(f"- Chunk Size: {CHUNK_SIZE}")
print(f"- Overlap: {OVERLAP}")

Configuration:
- PDF Directory: .\pdfContext
- Embedding Model: all-MiniLM-L6-v2
- Chunk Size: 1000
- Overlap: 200


## 2. Process PDFs and Build Index

In [2]:
def process_all_pdfs(pdf_dir=PDF_DIR, chunk_size=CHUNK_SIZE, overlap=OVERLAP, output_dir=OUTPUT_DIR):
    pdf_files = glob.glob(os.path.join(pdf_dir, '*.pdf'))
    
    if not pdf_files:
        print(f"No PDF files found in {pdf_dir}")
        return []
    
    all_chunks = []
    for pdf_path in pdf_files:
        print(f"Processing: {os.path.basename(pdf_path)}")
        chunks = process_pdf(pdf_path, chunk_size, overlap, output_dir)
        all_chunks.extend(chunks)
    
    return all_chunks

# Process PDFs
print("Processing PDFs...")
chunks = process_all_pdfs()
print(f"Total chunks created: {len(chunks)}")

Processing PDFs...
Processing: RNN Memory Caching.pdf
Total chunks created: 114


In [3]:
# Initialize RAG Pipeline
print("Initializing RAG Pipeline...")
rag = RAGPipeline(model_name=EMBEDDING_MODEL)

# Build index
print("Building embeddings and FAISS index...")
chunks_with_embeddings = rag.build_index(chunks)
print(f"Index built successfully with {len(chunks_with_embeddings)} chunks")
print(f"Embedding dimension: {rag.embedding_model.get_dimension()}")

Initializing RAG Pipeline...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building embeddings and FAISS index...
Index built successfully with 114 chunks
Embedding dimension: 384


## 3. Interactive Query Interface

**Change the query below and run the cell to get relevant chunks:**

In [4]:
# 🔍 CHANGE YOUR QUERY HERE
query = "What is Memory Caching?"
k = 5  # Number of results to return

# Search
print(f"Query: '{query}'")
print(f"Searching for top {k} relevant chunks...\n")

results = rag.query(query, k=k)

# Display results
print(f"Found {len(results)} results:\n")
print("=" * 80)

for i, result in enumerate(results, 1):
    chunk = result["chunk"]
    distance = result["distance"]
    
    print(f"\n🔸 Result {i} [Similarity Score: {1/(1+distance):.4f}]")
    print(f"📄 Source: {chunk['source']} (Page {chunk['page']})")
    print(f"🆔 Chunk ID: {chunk['chunk_id']}")
    print(f"📝 Text: {chunk['text'][:300]}{'...' if len(chunk['text']) > 300 else ''}")
    
    if chunk['images']:
        print(f"🖼️  Images: {len(chunk['images'])} image(s) found")
    
    print("-" * 80)

Query: 'What is Memory Caching?'
Searching for top 5 relevant chunks...

Found 5 results:


🔸 Result 1 [Similarity Score: 0.6149]
📄 Source: Memory Caching: RNNs with Growing Memory (Page 6)
🆔 Chunk ID: memory_caching:_rnns_with_growing_memory_29
📝 Text: he corresponding information toq t ins-th segment. To avoid interference of memories in different segments, we use independent memories for each segment, meaning that for segments, the memory starts from an initial pointM (s) 0 (·)that is independent ofM (s−1) L(s−1) (·). In practice, we observe tha...
--------------------------------------------------------------------------------

🔸 Result 2 [Similarity Score: 0.6130]
📄 Source: Memory Caching: RNNs with Growing Memory (Page 11)
🆔 Chunk ID: memory_caching:_rnns_with_growing_memory_56
📝 Text: e over their baseline. This shows the importance of memory caching to further enhance memory bounded models. (2) As discussed ear11
-----------------------------------------------------------------

## 4. Quick Query Function

Use this cell for rapid testing with different queries:

In [5]:
def quick_query(query_text, num_results=3):
    """Quick query function for testing"""
    results = rag.query(query_text, k=num_results)
    
    print(f"🔍 Query: '{query_text}'")
    print(f"📊 Top {len(results)} results:\n")
    
    for i, result in enumerate(results, 1):
        chunk = result["chunk"]
        score = 1/(1+result["distance"])
        
        print(f"{i}. [{score:.3f}] {chunk['source']} (Page {chunk['page']})")
        print(f"   {chunk['text'][:150]}...\n")

# Test different queries here:
quick_query("neural networks")
# quick_query("machine learning")
# quick_query("algorithms")

🔍 Query: 'neural networks'
📊 Top 3 results:

1. [0.468] Memory Caching: RNNs with Growing Memory (Page 22)
   rk by Hopfield (1982) introduced Hopfield Networks as one of the earliest neural architectures explicitly based on associative memory, formalized thro...

2. [0.467] Memory Caching: RNNs with Growing Memory (Page 19)
   s. InInternational Conference on Machine Learning, pp. 9355–9366. PMLR, 2021. Juergen Schmidhuber. Learning to control fast-weight memories: An altern...

3. [0.466] Memory Caching: RNNs with Growing Memory (Page 19)
   s, Quoc Le, Geoffrey Hinton, and Jeff Dean. Outrageously large neural networks: The sparsely-gated mixture-of-experts layer. InInternational Conferenc...



## 5. Advanced Search Options

In [6]:
def advanced_search(query_text, k=5, min_score=0.5):
    """Advanced search with filtering"""
    results = rag.query(query_text, k=k)
    
    # Filter by minimum similarity score
    filtered_results = []
    for result in results:
        score = 1/(1+result["distance"])
        if score >= min_score:
            result["similarity_score"] = score
            filtered_results.append(result)
    
    print(f"🔍 Query: '{query_text}'")
    print(f"📊 Found {len(filtered_results)} results above {min_score} similarity threshold:\n")
    
    for i, result in enumerate(filtered_results, 1):
        chunk = result["chunk"]
        score = result["similarity_score"]
        
        print(f"🔸 Result {i} [Score: {score:.4f}]")
        print(f"📄 {chunk['source']} (Page {chunk['page']})")
        print(f"📝 {chunk['text'][:200]}...\n")
    
    return filtered_results

# Example usage:
# advanced_search("memory optimization", k=10, min_score=0.6)

## 6. LLM Answer Generation via Ollama (Mistral 7B Q4)

In [7]:
import ollama
from config import GENERATION_MODEL, OLLAMA_URL

client = ollama.Client(host=OLLAMA_URL)

def ask(query_text, k=5):
    results = rag.query(query_text, k=k)
    context = "\n\n".join(r['chunk']['text'] for r in results)
    prompt = f"Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query_text}\nAnswer:"
    response = client.chat(
        model=GENERATION_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content']

# 💬 Change your question here
answer = ask("What is Memory Caching?")
print(answer)

Memory caching refers to a technique used in artificial intelligence and neural networks to store previously retrieved data in memory for faster access in future iterations. It involves dividing data into segments and storing each segment separately in memory, allowing for independent computation of the memory update and output computation at each timestep t. Memory caching can improve the performance of recurrent models by providing faster access to past memories, enhancing their ability to retain information over long periods. There are two main types of memory caching: linear memory modules and deep memory modules. In linear memory modules, the model uses a fixed-size memory that is updated based on the query, while in deep memory modules, the model uses a hierarchy of cached memories that can be accessed based on the query.


## 7. Evaluate RAG Response with Gemini

In [8]:
import json
import csv
import re
from datetime import datetime
from google import genai
from config import GOOGLE_API_KEY, EVALUATION_MODEL

eval_client = genai.Client(api_key=GOOGLE_API_KEY)

EVAL_PROMPT = """You are an evaluation judge for a RAG system.

Given:
- Question: {question}
- Retrieved Context: {context}  
- Generated Answer: {answer}

Score the answer on these 2 metrics:

1. Faithfulness (1-5): Is the answer factually grounded in the context?
2. Answer Relevancy (1-5): Does the answer actually address the question?

Respond in this JSON format only:
{{
  "faithfulness": <score>,
  "faithfulness_reason": "<one sentence>",
  "answer_relevancy": <score>,
  "answer_relevancy_reason": "<one sentence>"
}}"""

def evaluate(question, context, answer):
    prompt = EVAL_PROMPT.format(question=question, context=context, answer=answer)
    response = eval_client.models.generate_content(model=EVALUATION_MODEL, contents=prompt)
    raw = response.text.strip()
    json_str = re.search(r'\{.*\}', raw, re.DOTALL).group()
    return json.loads(json_str)

def evaluate_and_save(question, k=5, csv_path="eval_results.csv"):
    results = rag.query(question, k=k)
    context = "\n\n".join(r['chunk']['text'] for r in results)
    answer = ask(question, k=k)
    scores = evaluate(question, context, answer)

    row = {
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "context": context,
        "answer": answer,
        **scores
    }

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    print(f"Faithfulness    : {scores['faithfulness']}/5 - {scores['faithfulness_reason']}")
    print(f"Answer Relevancy: {scores['answer_relevancy']}/5 - {scores['answer_relevancy_reason']}")
    print(f"Saved to {csv_path}")
    return row

# 💬 Change your question here
evaluate_and_save("What is Memory Caching?")

Faithfulness    : 4/5 - The answer accurately describes memory caching using details from the context regarding online and cached memories and its variants, though some efficiency benefits are inferred rather than explicitly stated in the provided text.
Answer Relevancy: 5/5 - The answer directly defines and explains memory caching, its mechanism, and its variants, completely addressing the 'what' in the question.
Saved to eval_results.csv


{'timestamp': '2026-06-10T06:58:16.839856',
 'question': 'What is Memory Caching?',
 'context': 'he corresponding information toq t ins-th segment. To avoid interference of memories in different segments, we use independent memories for each segment, meaning that for segments, the memory starts from an initial pointM (s) 0 (·)that is independent ofM (s−1) L(s−1) (·). In practice, we observe that each of these two choices has its own (dis)advantageous (see Section 5.6). 4 DISCUSSION ANDPROOF OFCONCEPT In this section we discuss the implication of using memory caching technique for linear and deep memory modules, and discuss the models we use as a proof of concept in our evaluations. 6\n\ne over their baseline. This shows the importance of memory caching to further enhance memory bounded models. (2) As discussed ear11\n\nat enforces caching past inputs. Note that the above explanation demonstrates an equivalency only for an oversimplified version, and with considering normalization and f

## 8. Save/Load Index (Optional)

In [9]:
# Save the index for future use
index_path = "faiss_index.bin"
rag.save_index(index_path)
print(f"Index saved to {index_path}")

# To load later:
# rag.load_index(index_path, chunks)
# print("Index loaded successfully")

Index saved to faiss_index.bin
